In [ ]:
python3 -c "
import base64, ssl
from pyVim.connect import SmartConnect, Disconnect
from pyVmomi import vim

context = ssl.SSLContext(ssl.PROTOCOL_TLS_CLIENT)
context.check_hostname = False
context.verify_mode = ssl.CERT_NONE

si = SmartConnect(host='172.20.0.101', user='administrator@vsphere.local', pwd='Root@123', sslContext=context)
content = si.RetrieveContent()

view = content.viewManager.CreateContainerView(content.rootFolder, [vim.VirtualMachine], True)
vm = next((v for v in view.view if v.name == 'ocp4-bootstrap'), None)

with open('$OCP4_DIR/config/bootstrap.ign', 'rb') as f:
    ign_data = base64.b64encode(f.read()).decode('utf-8')

spec = vim.vm.ConfigSpec()
spec.extraConfig = [
    vim.option.OptionValue(key='guestinfo.ignition.config.data', value=ign_data),
    vim.option.OptionValue(key='guestinfo.ignition.config.data.encoding', value='base64')
]
task = vm.ReconfigVM_Task(spec)
while task.info.state == vim.TaskInfo.State.running:
    pass
print('Result:', task.info.state)

Disconnect(si)
"

In [ ]:
opencode -s ses_0ec5eb407ffeCjrNIGHlbvYYk3

In [ ]:
You should create cluster and put the ESXIs on it.

create 

In [ ]:
cat > install-config.yaml <<'EOF'
apiVersion: v1
baseDomain: example.com
metadata:
  name: ocp4
compute:
- architecture: amd64
  hyperthreading: Enabled
  name: worker
  platform: {}
  replicas: 0
controlPlane:
  architecture: amd64
  hyperthreading: Enabled
  name: master
  platform: {}
  replicas: 3
platform:
  vsphere:
    apiVIPs:
    - 172.20.0.220
    ingressVIPs:
    - 172.20.0.221
    vcenters:
    - datacenters:
      - Datacenter
      server: 172.20.0.101
      user: administrator@vsphere.local
      password: Root@123
      port: 443
    failureDomains:
    - name: fd-1
      region: region-1
      server: 172.20.0.101
      topology:
        computeCluster: /Datacenter/host/Openshift
        datacenter: Datacenter
        datastore: /Datacenter/datastore/hdd
        networks:
        - VM Network
        resourcePool: /Datacenter/host/Openshift/Resources
        folder: /Datacenter/vm/
      zone: zone-1
networking:
  clusterNetwork:
  - cidr: 10.128.0.0/14
    hostPrefix: 23
  machineNetwork:
  - cidr: 172.20.0.0/24
  networkType: OVNKubernetes
  serviceNetwork:
  - 172.30.0.0/16
publish: External
pullSecret: '{"auths":{"cloud.openshift.com":{"auth":"b3BlbnNoaWZ0LXJlbGVhc2UtZGV2K29jbV9hY2Nlc3NfYjhhMjA1NjhkNzViNGNkN2IwZmE3ZmU1YzQ1MmIxY2U6UEpWRVpSWFk3MzExNU41V0NFNERXRlBSMFJQOUpGWVgyRzMwWlAzRVRKVjgzVUREOE02VzBURlIyT0ROWEFXQQ==","email":"khlaedmohamedeldsoky@gmail.com"},"quay.io":{"auth":"b3BlbnNoaWZ0LXJlbGVhc2UtZGV2K29jbV9hY2Nlc3NfYjhhMjA1NjhkNzViNGNkN2IwZmE3ZmU1YzQ1MmIxY2U6UEpWRVpSWFk3MzExNU41V0NFNERXRlBSMFJQOUpGWVgyRzMwWlAzRVRKVjgzVUREOE02VzBURlIyT0ROWEFXQQ==","email":"khlaedmohamedeldsoky@gmail.com"},"registry.connect.redhat.com":{"auth":"fHVoYy1wb29sLWJmNmE2MDVlLTFmYzgtNDhmMi1iOTFjLWFhODMxMjlkY2M1ZTpleUpoYkdjaU9pSlNVelV4TWlKOS5leUp6ZFdJaU9pSmpPRE16WkRnd1pERmxPVEEwTURVeFltRXdNVFF6T1RNMFpqVTNORGxpT0NKOS5hWHZpUmlOX2pqeFUxN1BTUVBrSUVwU0N2U1h5UmhpQTVjYi1wcmxjaEotUlRvQk1oTi1kWVN4emk0TjlJdVNkSjlJZU5BUU45ZjF1NC1qeUpzSWFoeDBzN2h2UTdubzI5aXFlZmV6cTl4eXRPRHhyWGRfZi1qQ0NIMlZQbEFFOF9NcFBLdWJCRVVTUWR0NHJpeVBIdUROaWRUVV80dTRzOExFMUFTb3JlNElpUmNaczhGTU5UcmdTUkVldEd4cGZIenFlZDFLSl9NMXN2RVE2WEJwUU93WTJvZkwyMkVRaGFoRDJhbzR4cG9YelNHOTZDZUtvY1k5UjJCSFV4YzEyejByellBS054V1R5OHdoZUl6Qmk4cWFvUVhUYV9CY3BQaERIdzA2ejZzdUVrekk1NFJEMXJMTWJaNXFqb05OLTU3QXB6ZURYUTk5Q1N1Q0pQWjF1QUVNc01MMkltd0QweDc2TkwxbjZUSUNtVmpBZC04bUJOakVrbnViOGpzZXc2TGRNZHFWLW1ucDRSMmdCS0FSN29kZTV1bFZEWldvOG50Z3lkOGNGVy1ic0hLc2RMenRSX1k1T1hKTXlteE5na2RWb3lDbmlQT29razNtODl0dF91UEVFUGd1MEZndVBxaDVtVlNWeldXWFFNLVFRc0pKUEJxUnJrNURvaWxuRzJicGdvZE1rdlF5VFFDQUtwR3ZSWGh6NWFkQ3F0ZVgwMEVKSHF5bGZ3X1VpbnFtbW5SWjdHMU5aZU1pdnpydjAtZXpKdHFqWEtrdGw0dEZIWmVZeU91b3FndnQ3NzVOamlWNkU2RTJwNDRHZEJhR1FKRmJxS1g1YVRRRHZseW5YMnZMZ0FmSlBySWZyTTE4TS1yV3BSU3F0Nng3OXNwb0xZRTJoelcyWVJITQ==","email":"khlaedmohamedeldsoky@gmail.com"},"registry.redhat.io":{"auth":"fHVoYy1wb29sLWJmNmE2MDVlLTFmYzgtNDhmMi1iOTFjLWFhODMxMjlkY2M1ZTpleUpoYkdjaU9pSlNVelV4TWlKOS5leUp6ZFdJaU9pSmpPRE16WkRnd1pERmxPVEEwTURVeFltRXdNVFF6T1RNMFpqVTNORGxpT0NKOS5hWHZpUmlOX2pqeFUxN1BTUVBrSUVwU0N2U1h5UmhpQTVjYi1wcmxjaEotUlRvQk1oTi1kWVN4emk0TjlJdVNkSjlJZU5BUU45ZjF1NC1qeUpzSWFoeDBzN2h2UTdubzI5aXFlZmV6cTl4eXRPRHhyWGRfZi1qQ0NIMlZQbEFFOF9NcFBLdWJCRVVTUWR0NHJpeVBIdUROaWRUVV80dTRzOExFMUFTb3JlNElpUmNaczhGTU5UcmdTUkVldEd4cGZIenFlZDFLSl9NMXN2RVE2WEJwUU93WTJvZkwyMkVRaGFoRDJhbzR4cG9YelNHOTZDZUtvY1k5UjJCSFV4YzEyejByellBS054V1R5OHdoZUl6Qmk4cWFvUVhUYV9CY3BQaERIdzA2ejZzdUVrekk1NFJEMXJMTWJaNXFqb05OLTU3QXB6ZURYUTk5Q1N1Q0pQWjF1QUVNc01MMkltd0QweDc2TkwxbjZUSUNtVmpBZC04bUJOakVrbnViOGpzZXc2TGRNZHFWLW1ucDRSMmdCS0FSN29kZTV1bFZEWldvOG50Z3lkOGNGVy1ic0hLc2RMenRSX1k1T1hKTXlteE5na2RWb3lDbmlQT29razNtODl0dF91UEVFUGd1MEZndVBxaDVtVlNWeldXWFFNLVFRc0pKUEJxUnJrNURvaWxuRzJicGdvZE1rdlF5VFFDQUtwR3ZSWGh6NWFkQ3F0ZVgwMEVKSHF5bGZ3X1VpbnFtbW5SWjdHMU5aZU1pdnpydjAtZXpKdHFqWEtrdGw0dEZIWmVZeU91b3FndnQ3NzVOamlWNkU2RTJwNDRHZEJhR1FKRmJxS1g1YVRRRHZseW5YMnZMZ0FmSlBySWZyTTE4TS1yV3BSU3F0Nng3OXNwb0xZRTJoelcyWVJITQ==","email":"khlaedmohamedeldsoky@gmail.com"}}}'
sshKey: |
  ssh-ed25519 AAAAC3NzaC1lZDI1NTE5AAAAIL+Go0IWOR7LQoVNbz8DooJ6BwXvuG3Dzi/oA20fJ/s4 khaled@BI-K-Eldsouky
EOF

the last use

In [ ]:
cat > install-config.yaml <<'EOF'
apiVersion: v1
baseDomain: example.com
metadata:
  name: ocp4
compute:
- architecture: amd64
  hyperthreading: Enabled
  name: worker
  platform: {}
  replicas: 0
controlPlane:
  architecture: amd64
  hyperthreading: Enabled
  name: master
  platform: {}
  replicas: 3
platform:
  vsphere:
    vcenter: 172.20.0.101
    username: administrator@vsphere.local
    password: Root@123
    datacenter: Datacenter
    defaultDatastore: hdd
    folder: /Datacenter/vm/
    network: VM Network
    cluster: Openshift
    resourcePool: /Datacenter/host/Openshift/Resources
networking:
  clusterNetwork:
  - cidr: 10.128.0.0/14
    hostPrefix: 23
  machineNetwork:
  - cidr: 172.20.0.0/24
  networkType: OVNKubernetes
  serviceNetwork:
  - 172.30.0.0/16
publish: External
pullSecret: '{"auths":{"cloud.openshift.com":{"auth":"b3BlbnNoaWZ0LXJlbGVhc2UtZGV2K29jbV9hY2Nlc3NfYjhhMjA1NjhkNzViNGNkN2IwZmE3ZmU1YzQ1MmIxY2U6UEpWRVpSWFk3MzExNU41V0NFNERXRlBSMFJQOUpGWVgyRzMwWlAzRVRKVjgzVUREOE02VzBURlIyT0ROWEFXQQ==","email":"khlaedmohamedeldsoky@gmail.com"},"quay.io":{"auth":"b3BlbnNoaWZ0LXJlbGVhc2UtZGV2K29jbV9hY2Nlc3NfYjhhMjA1NjhkNzViNGNkN2IwZmE3ZmU1YzQ1MmIxY2U6UEpWRVpSWFk3MzExNU41V0NFNERXRlBSMFJQOUpGWVgyRzMwWlAzRVRKVjgzVUREOE02VzBURlIyT0ROWEFXQQ==","email":"khlaedmohamedeldsoky@gmail.com"},"registry.connect.redhat.com":{"auth":"fHVoYy1wb29sLWJmNmE2MDVlLTFmYzgtNDhmMi1iOTFjLWFhODMxMjlkY2M1ZTpleUpoYkdjaU9pSlNVelV4TWlKOS5leUp6ZFdJaU9pSmpPRE16WkRnd1pERmxPVEEwTURVeFltRXdNVFF6T1RNMFpqVTNORGxpT0NKOS5hWHZpUmlOX2pqeFUxN1BTUVBrSUVwU0N2U1h5UmhpQTVjYi1wcmxjaEotUlRvQk1oTi1kWVN4emk0TjlJdVNkSjlJZU5BUU45ZjF1NC1qeUpzSWFoeDBzN2h2UTdubzI5aXFlZmV6cTl4eXRPRHhyWGRfZi1qQ0NIMlZQbEFFOF9NcFBLdWJCRVVTUWR0NHJpeVBIdUROaWRUVV80dTRzOExFMUFTb3JlNElpUmNaczhGTU5UcmdTUkVldEd4cGZIenFlZDFLSl9NMXN2RVE2WEJwUU93WTJvZkwyMkVRaGFoRDJhbzR4cG9YelNHOTZDZUtvY1k5UjJCSFV4YzEyejByellBS054V1R5OHdoZUl6Qmk4cWFvUVhUYV9CY3BQaERIdzA2ejZzdUVrekk1NFJEMXJMTWJaNXFqb05OLTU3QXB6ZURYUTk5Q1N1Q0pQWjF1QUVNc01MMkltd0QweDc2TkwxbjZUSUNtVmpBZC04bUJOakVrbnViOGpzZXc2TGRNZHFWLW1ucDRSMmdCS0FSN29kZTV1bFZEWldvOG50Z3lkOGNGVy1ic0hLc2RMenRSX1k1T1hKTXlteE5na2RWb3lDbmlQT29razNtODl0dF91UEVFUGd1MEZndVBxaDVtVlNWeldXWFFNLVFRc0pKUEJxUnJrNURvaWxuRzJicGdvZE1rdlF5VFFDQUtwR3ZSWGh6NWFkQ3F0ZVgwMEVKSHF5bGZ3X1VpbnFtbW5SWjdHMU5aZU1pdnpydjAtZXpKdHFqWEtrdGw0dEZIWmVZeU91b3FndnQ3NzVOamlWNkU2RTJwNDRHZEJhR1FKRmJxS1g1YVRRRHZseW5YMnZMZ0FmSlBySWZyTTE4TS1yV3BSU3F0Nng3OXNwb0xZRTJoelcyWVJITQ==","email":"khlaedmohamedeldsoky@gmail.com"},"registry.redhat.io":{"auth":"fHVoYy1wb29sLWJmNmE2MDVlLTFmYzgtNDhmMi1iOTFjLWFhODMxMjlkY2M1ZTpleUpoYkdjaU9pSlNVelV4TWlKOS5leUp6ZFdJaU9pSmpPRE16WkRnd1pERmxPVEEwTURVeFltRXdNVFF6T1RNMFpqVTNORGxpT0NKOS5hWHZpUmlOX2pqeFUxN1BTUVBrSUVwU0N2U1h5UmhpQTVjYi1wcmxjaEotUlRvQk1oTi1kWVN4emk0TjlJdVNkSjlJZU5BUU45ZjF1NC1qeUpzSWFoeDBzN2h2UTdubzI5aXFlZmV6cTl4eXRPRHhyWGRfZi1qQ0NIMlZQbEFFOF9NcFBLdWJCRVVTUWR0NHJpeVBIdUROaWRUVV80dTRzOExFMUFTb3JlNElpUmNaczhGTU5UcmdTUkVldEd4cGZIenFlZDFLSl9NMXN2RVE2WEJwUU93WTJvZkwyMkVRaGFoRDJhbzR4cG9YelNHOTZDZUtvY1k5UjJCSFV4YzEyejByellBS054V1R5OHdoZUl6Qmk4cWFvUVhUYV9CY3BQaERIdzA2ejZzdUVrekk1NFJEMXJMTWJaNXFqb05OLTU3QXB6ZURYUTk5Q1N1Q0pQWjF1QUVNc01MMkltd0QweDc2TkwxbjZUSUNtVmpBZC04bUJOakVrbnViOGpzZXc2TGRNZHFWLW1ucDRSMmdCS0FSN29kZTV1bFZEWldvOG50Z3lkOGNGVy1ic0hLc2RMenRSX1k1T1hKTXlteE5na2RWb3lDbmlQT29razNtODl0dF91UEVFUGd1MEZndVBxaDVtVlNWeldXWFFNLVFRc0pKUEJxUnJrNURvaWxuRzJicGdvZE1rdlF5VFFDQUtwR3ZSWGh6NWFkQ3F0ZVgwMEVKSHF5bGZ3X1VpbnFtbW5SWjdHMU5aZU1pdnpydjAtZXpKdHFqWEtrdGw0dEZIWmVZeU91b3FndnQ3NzVOamlWNkU2RTJwNDRHZEJhR1FKRmJxS1g1YVRRRHZseW5YMnZMZ0FmSlBySWZyTTE4TS1yV3BSU3F0Nng3OXNwb0xZRTJoelcyWVJITQ==","email":"khlaedmohamedeldsoky@gmail.com"}}}'
sshKey: |
  ssh-ed25519 AAAAC3NzaC1lZDI1NTE5AAAAIL+Go0IWOR7LQoVNbz8DooJ6BwXvuG3Dzi/oA20fJ/s4 khaled@BI-K-Eldsouky
EOF

In [ ]:
export GOVC_URL=172.20.0.101
export GOVC_USERNAME=administrator@vsphere.local
export GOVC_PASSWORD=Root@123
export GOVC_INSECURE=1
export GOVC_DATACENTER=Datacenter
export GOVC_DATASTORE=hdd
export GOVC_NETWORK="VM Network"
export GOVC_RESOURCE_POOL="/Datacenter/host/Openshift/Resources"

In [ ]:
mkdir ~/ocp4-install && cd ~/ocp4-install
./openshift-install create manifests --dir=.
./openshift-install create ignition-configs --dir=.

In [ ]:
govc about
govc ls /Datacenter/vm

In [ ]:
cat >  create_ocp_vms.sh <<EOF
#!/bin/bash
# create_ocp_vms.sh

cd ~/ocp4-install

GATEWAY="172.20.0.254"
NETMASK="255.255.255.0"
DNS="172.16.6.70"

# ==================== Prepare Ignition Files ====================
echo "=== Encoding Ignition Files ==="
cat bootstrap.ign | base64 -w0 > bootstrap.ign.b64
cat master.ign | base64 -w0 > master.ign.b64
cat worker.ign | base64 -w0 > worker.ign.b64
echo "✓ Done"
echo ""

# ==================== Bootstrap ====================
echo "=== Creating ocp4-bootstrap ==="
govc vm.clone -vm=rhcos-template -on=false ocp4-bootstrap
govc vm.change -vm=ocp4-bootstrap -c=4 -m=16384
govc vm.disk.change -vm=ocp4-bootstrap -size 120G
govc vm.change -e="disk.EnableUUID=TRUE" -vm=ocp4-bootstrap
cat bootstrap.ign.b64 | xargs -I {} govc vm.change -e="guestinfo.ignition.config.data={}" -vm=ocp4-bootstrap
govc vm.change -e="guestinfo.ignition.config.data.encoding=base64" -vm=ocp4-bootstrap
govc vm.change -e="guestinfo.afterburn.initrd.network-kargs=ip=172.20.0.230::${GATEWAY}:${NETMASK}:bootstrap.ocp4.example.com:ens192:off nameserver=${DNS}" -vm=ocp4-bootstrap
govc vm.power -on ocp4-bootstrap
sleep 120

# ==================== Masters ====================
echo "=== Creating ocp4-master-0 ==="
govc vm.clone -vm=rhcos-template -on=false ocp4-master-0
govc vm.change -vm=ocp4-master-0 -c=4 -m=16384
govc vm.disk.change -vm=ocp4-master-0 -size 120G
govc vm.change -e="disk.EnableUUID=TRUE" -vm=ocp4-master-0
cat master.ign.b64 | xargs -I {} govc vm.change -e="guestinfo.ignition.config.data={}" -vm=ocp4-master-0
govc vm.change -e="guestinfo.ignition.config.data.encoding=base64" -vm=ocp4-master-0
govc vm.change -e="guestinfo.afterburn.initrd.network-kargs=ip=172.20.0.231::${GATEWAY}:${NETMASK}:master-0.ocp4.example.com:ens192:off nameserver=${DNS}" -vm=ocp4-master-0
govc vm.power -on ocp4-master-0

echo "=== Creating ocp4-master-1 ==="
govc vm.clone -vm=rhcos-template -on=false ocp4-master-1
govc vm.change -vm=ocp4-master-1 -c=4 -m=16384
govc vm.disk.change -vm=ocp4-master-1 -size 120G
govc vm.change -e="disk.EnableUUID=TRUE" -vm=ocp4-master-1
cat master.ign.b64 | xargs -I {} govc vm.change -e="guestinfo.ignition.config.data={}" -vm=ocp4-master-1
govc vm.change -e="guestinfo.ignition.config.data.encoding=base64" -vm=ocp4-master-1
govc vm.change -e="guestinfo.afterburn.initrd.network-kargs=ip=172.20.0.232::${GATEWAY}:${NETMASK}:master-1.ocp4.example.com:ens192:off nameserver=${DNS}" -vm=ocp4-master-1
govc vm.power -on ocp4-master-1

echo "=== Creating ocp4-master-2 ==="
govc vm.clone -vm=rhcos-template -on=false ocp4-master-2
govc vm.change -vm=ocp4-master-2 -c=4 -m=16384
govc vm.disk.change -vm=ocp4-master-2 -size 120G
govc vm.change -e="disk.EnableUUID=TRUE" -vm=ocp4-master-2
cat master.ign.b64 | xargs -I {} govc vm.change -e="guestinfo.ignition.config.data={}" -vm=ocp4-master-2
govc vm.change -e="guestinfo.ignition.config.data.encoding=base64" -vm=ocp4-master-2
govc vm.change -e="guestinfo.afterburn.initrd.network-kargs=ip=172.20.0.233::${GATEWAY}:${NETMASK}:master-2.ocp4.example.com:ens192:off nameserver=${DNS}" -vm=ocp4-master-2
govc vm.power -on ocp4-master-2

sleep 60

# ==================== Workers ====================
echo "=== Creating ocp4-worker-0 ==="
govc vm.clone -vm=rhcos-template -on=false ocp4-worker-0
govc vm.change -vm=ocp4-worker-0 -c=2 -m=8192
govc vm.disk.change -vm=ocp4-worker-0 -size 120G
govc vm.change -e="disk.EnableUUID=TRUE" -vm=ocp4-worker-0
cat worker.ign.b64 | xargs -I {} govc vm.change -e="guestinfo.ignition.config.data={}" -vm=ocp4-worker-0
govc vm.change -e="guestinfo.ignition.config.data.encoding=base64" -vm=ocp4-worker-0
govc vm.change -e="guestinfo.afterburn.initrd.network-kargs=ip=172.20.0.234::${GATEWAY}:${NETMASK}:worker-0.ocp4.example.com:ens192:off nameserver=${DNS}" -vm=ocp4-worker-0
govc vm.power -on ocp4-worker-0

echo "=== Creating ocp4-worker-1 ==="
govc vm.clone -vm=rhcos-template -on=false ocp4-worker-1
govc vm.change -vm=ocp4-worker-1 -c=2 -m=8192
govc vm.disk.change -vm=ocp4-worker-1 -size 120G
govc vm.change -e="disk.EnableUUID=TRUE" -vm=ocp4-worker-1
cat worker.ign.b64 | xargs -I {} govc vm.change -e="guestinfo.ignition.config.data={}" -vm=ocp4-worker-1
govc vm.change -e="guestinfo.ignition.config.data.encoding=base64" -vm=ocp4-worker-1
govc vm.change -e="guestinfo.afterburn.initrd.network-kargs=ip=172.20.0.235::${GATEWAY}:${NETMASK}:worker-1.ocp4.example.com:ens192:off nameserver=${DNS}" -vm=ocp4-worker-1
govc vm.power -on ocp4-worker-1

echo ""
echo "=== All VMs Created ==="
echo "ocp4-bootstrap: 172.20.0.230"
echo "ocp4-master-0:  172.20.0.231"
echo "ocp4-master-1:  172.20.0.232"
echo "ocp4-master-2:  172.20.0.233"
echo "ocp4-worker-0:  172.20.0.234"
echo "ocp4-worker-1:  172.20.0.235"
EOF


In [ ]:
./openshift-install create manifests --dir=.
./openshift-install create ignition-configs --dir=.

In [ ]:
cat > /tmp/fix_ignition.py <<'PYEOF'
#!/usr/bin/env python3
import base64
import ssl
from pyVim.connect import SmartConnect, Disconnect
from pyVmomi import vim

VCENTER = '172.20.0.101'
USERNAME = 'administrator@vsphere.local'
PASSWORD = 'Root@123'  # غيّر ده!

vms = {
    'ocp4-bootstrap': 'bootstrap.ign',
    'ocp4-master-0': 'master.ign',
    'ocp4-master-1': 'master.ign',
    'ocp4-master-2': 'master.ign',
    'ocp4-worker-0': 'worker.ign',
    'ocp4-worker-1': 'worker.ign',
}

context = ssl.SSLContext(ssl.PROTOCOL_TLS_CLIENT)
context.check_hostname = False
context.verify_mode = ssl.CERT_NONE

si = SmartConnect(host=VCENTER, user=USERNAME, pwd=PASSWORD, sslContext=context)
content = si.RetrieveContent()

for vm_name, ign_file in vms.items():
    print(f"\nProcessing {vm_name}...")
    
    # Find VM
    obj_view = content.viewManager.CreateContainerView(content.rootFolder, [vim.VirtualMachine], True)
    vm = None
    for v in obj_view.view:
        if v.name == vm_name:
            vm = v
            break
    obj_view.Destroy()
    
    if not vm:
        print(f"  ERROR: {vm_name} not found!")
        continue
    
    # Read and encode ignition
    with open(ign_file, 'rb') as f:
        data = base64.b64encode(f.read()).decode('utf-8')
    
    # Set extraConfig
    spec = vim.vm.ConfigSpec()
    spec.extraConfig = [
        vim.option.OptionValue(key='guestinfo.ignition.config.data', value=data),
        vim.option.OptionValue(key='guestinfo.ignition.config.data.encoding', value='base64')
    ]
    
    task = vm.ReconfigVM_Task(spec)
    
    # Wait for task
    while task.info.state == 'running':
        import time
        time.sleep(1)
    
    if task.info.state == 'success':
        print(f"  ✓ {vm_name} done")
    else:
        print(f"  ✗ {vm_name} failed: {task.info.error}")

print("\nAll done!")
Disconnect(si)
PYEOF

cd ~/ocp4-install
python3 /tmp/fix_ignition.py

It wooooooooooooooork

In [ ]:
cat > create_ocp_vms.py <<EOF
#!/usr/bin/env python3
# create_ocp_vms.py

import base64
import ssl
from pyVim.connect import SmartConnect, Disconnect
from pyVmomi import vim

VCENTER = "172.20.0.101"
USERNAME = "administrator@vsphere.local"
PASSWORD = "Root@123"

GATEWAY = "172.20.0.254"
NETMASK = "255.255.255.0"
DNS = "172.16.6.70"

vms = {
    "ocp4-bootstrap": {
        "ip": "172.20.0.230",
        "hostname": "bootstrap.ocp4.example.com",
        "cpu": 4,
        "ram": 16384,
        "disk": 120,
        "ign": "bootstrap.ign"
    },
    "ocp4-master-0": {
        "ip": "172.20.0.231",
        "hostname": "master-0.ocp4.example.com",
        "cpu": 4,
        "ram": 16384,
        "disk": 120,
        "ign": "master.ign"
    },
    "ocp4-master-1": {
        "ip": "172.20.0.232",
        "hostname": "master-1.ocp4.example.com",
        "cpu": 4,
        "ram": 16384,
        "disk": 120,
        "ign": "master.ign"
    },
    "ocp4-master-2": {
        "ip": "172.20.0.233",
        "hostname": "master-2.ocp4.example.com",
        "cpu": 4,
        "ram": 16384,
        "disk": 120,
        "ign": "master.ign"
    },
    "ocp4-worker-0": {
        "ip": "172.20.0.234",
        "hostname": "worker-0.ocp4.example.com",
        "cpu": 2,
        "ram": 8192,
        "disk": 120,
        "ign": "worker.ign"
    },
    "ocp4-worker-1": {
        "ip": "172.20.0.235",
        "hostname": "worker-1.ocp4.example.com",
        "cpu": 2,
        "ram": 8192,
        "disk": 120,
        "ign": "worker.ign"
    }
}

context = ssl.SSLContext(ssl.PROTOCOL_TLS_CLIENT)
context.check_hostname = False
context.verify_mode = ssl.CERT_NONE

si = SmartConnect(
    host=VCENTER,
    user=USERNAME,
    pwd=PASSWORD,
    sslContext=context
)

content = si.RetrieveContent()


def get_vm(name):
    view = content.viewManager.CreateContainerView(
        content.rootFolder,
        [vim.VirtualMachine],
        True
    )

    for vm in view.view:
        if vm.name == name:
            return vm

    return None


def get_resource_pool():
    view = content.viewManager.CreateContainerView(
        content.rootFolder,
        [vim.ResourcePool],
        True
    )

    for rp in view.view:
        if rp.name == "Resources":
            return rp

    return None


resource_pool = get_resource_pool()

if resource_pool is None:
    print("ERROR: Resource Pool 'Resources' not found.")
    Disconnect(si)
    exit(1)


for vm_name, config in vms.items():

    print(f"Processing {vm_name}...")

    template = get_vm("rhcos-template")

    if template is None:
        print("ERROR: Template not found!")
        break

    clone_spec = vim.vm.CloneSpec()
    clone_spec.location = vim.vm.RelocateSpec()
    clone_spec.location.pool = resource_pool

    task = template.Clone(
        folder=template.parent,
        name=vm_name,
        spec=clone_spec
    )

    while task.info.state == vim.TaskInfo.State.running:
        pass

    if task.info.state == vim.TaskInfo.State.error:
        print(f"Clone failed for {vm_name}")
        print(task.info.error)
        continue

    vm = get_vm(vm_name)

    if vm is None:
        print(f"ERROR: Failed to find {vm_name} after clone")
        continue

    spec = vim.vm.ConfigSpec()
    spec.numCPUs = config["cpu"]
    spec.memoryMB = config["ram"]

    spec.extraConfig = [
        vim.option.OptionValue(
            key="disk.EnableUUID",
            value="TRUE"
        )
    ]

    task = vm.ReconfigVM_Task(spec)

    while task.info.state == vim.TaskInfo.State.running:
        pass

    if task.info.state == vim.TaskInfo.State.error:
        print(task.info.error)
        continue

    with open(config["ign"], "rb") as f:
        ign_data = base64.b64encode(f.read()).decode("utf-8")

    spec = vim.vm.ConfigSpec()

    spec.extraConfig = [
        vim.option.OptionValue(
            key="guestinfo.ignition.config.data",
            value=ign_data
        ),
        vim.option.OptionValue(
            key="guestinfo.ignition.config.data.encoding",
            value="base64"
        ),
        vim.option.OptionValue(
            key="guestinfo.afterburn.initrd.network-kargs",
            value=f"ip={config['ip']}::{GATEWAY}:{NETMASK}:{config['hostname']}:ens192:off nameserver={DNS}"
        )
    ]

    task = vm.ReconfigVM_Task(spec)

    while task.info.state == vim.TaskInfo.State.running:
        pass

    if task.info.state == vim.TaskInfo.State.error:
        print(task.info.error)
        continue

    print(f"✓ {vm_name} created")

Disconnect(si)

print("\nAll VMs created!")
print("Power on VMs in order: Bootstrap -> Masters -> Workers")
EOF

In [ ]:
python3 create_ocp_vms.py

The lst used

In [ ]:
cat > create_ocp_vms.py <<EOF
#!/usr/bin/env python3
# create_ocp_vms.py

import base64
import ssl
from pyVim.connect import SmartConnect, Disconnect
from pyVmomi import vim

VCENTER = "172.20.0.101"
USERNAME = "administrator@vsphere.local"
PASSWORD = "Root@123"

GATEWAY = "172.20.0.254"
NETMASK = "255.255.255.0"
DNS = "172.16.6.70"

vms = {
    "ocp4-bootstrap": {
        "ip": "172.20.0.230",
        "hostname": "bootstrap.ocp4.example.com",
        "cpu": 4,
        "ram": 16384,
        "disk": 120,  # GB
        "ign": "bootstrap.ign"
    },
    "ocp4-master-0": {
        "ip": "172.20.0.231",
        "hostname": "master-0.ocp4.example.com",
        "cpu": 4,
        "ram": 16384,
        "disk": 120,
        "ign": "master.ign"
    },
    "ocp4-master-1": {
        "ip": "172.20.0.232",
        "hostname": "master-1.ocp4.example.com",
        "cpu": 4,
        "ram": 16384,
        "disk": 120,
        "ign": "master.ign"
    },
    "ocp4-master-2": {
        "ip": "172.20.0.233",
        "hostname": "master-2.ocp4.example.com",
        "cpu": 4,
        "ram": 16384,
        "disk": 120,
        "ign": "master.ign"
    },
    "ocp4-worker-0": {
        "ip": "172.20.0.234",
        "hostname": "worker-0.ocp4.example.com",
        "cpu": 2,
        "ram": 8192,
        "disk": 120,
        "ign": "worker.ign"
    },
    "ocp4-worker-1": {
        "ip": "172.20.0.235",
        "hostname": "worker-1.ocp4.example.com",
        "cpu": 2,
        "ram": 8192,
        "disk": 120,
        "ign": "worker.ign"
    }
}

context = ssl.SSLContext(ssl.PROTOCOL_TLS_CLIENT)
context.check_hostname = False
context.verify_mode = ssl.CERT_NONE

si = SmartConnect(
    host=VCENTER,
    user=USERNAME,
    pwd=PASSWORD,
    sslContext=context
)

content = si.RetrieveContent()


def get_vm(name):
    view = content.viewManager.CreateContainerView(
        content.rootFolder,
        [vim.VirtualMachine],
        True
    )
    for vm in view.view:
        if vm.name == name:
            return vm
    return None


def get_resource_pool():
    view = content.viewManager.CreateContainerView(
        content.rootFolder,
        [vim.ResourcePool],
        True
    )
    for rp in view.view:
        if rp.name == "Resources":
            return rp
    return None


def wait_for_task(task):
    while task.info.state == vim.TaskInfo.State.running:
        pass
    if task.info.state == vim.TaskInfo.State.error:
        print(f"  ERROR: {task.info.error}")
        return False
    return True


resource_pool = get_resource_pool()
if resource_pool is None:
    print("ERROR: Resource Pool 'Resources' not found.")
    Disconnect(si)
    exit(1)

for vm_name, config in vms.items():
    print(f"\n=== Processing {vm_name} ===")

    template = get_vm("rhcos-template")
    if template is None:
        print("ERROR: Template not found!")
        break

    # 1. Clone VM
    print(f"  Cloning from template...")
    clone_spec = vim.vm.CloneSpec()
    clone_spec.location = vim.vm.RelocateSpec()
    clone_spec.location.pool = resource_pool

    task = template.Clone(folder=template.parent, name=vm_name, spec=clone_spec)
    if not wait_for_task(task):
        continue

    vm = get_vm(vm_name)
    if vm is None:
        print(f"ERROR: Failed to find {vm_name} after clone")
        continue

    # 2. Reconfigure CPU, RAM, and DISK
    print(f"  Setting CPU={config['cpu']}, RAM={config['ram']}MB, DISK={config['disk']}GB...")
    
    spec = vim.vm.ConfigSpec()
    spec.numCPUs = config["cpu"]
    spec.memoryMB = config["ram"]

    # Resize the first hard disk (index 0) to the specified size
    disk_size_gb = config["disk"]
    disk_size_kb = disk_size_gb * 1024 * 1024  # Convert GB to KB

    # Find the first virtual disk device
    disk_device = None
    for device in vm.config.hardware.device:
        if isinstance(device, vim.vm.device.VirtualDisk):
            disk_device = device
            break

    if disk_device is None:
        print(f"  WARNING: No virtual disk found on {vm_name}, skipping disk resize")
    else:
        disk_spec = vim.vm.device.VirtualDeviceSpec()
        disk_spec.operation = vim.vm.device.VirtualDeviceSpec.Operation.edit
        disk_spec.device = disk_device
        disk_spec.device.capacityInKB = disk_size_kb
        spec.deviceChange = [disk_spec]

    # Add disk.EnableUUID for CSI
    spec.extraConfig = [
        vim.option.OptionValue(key="disk.EnableUUID", value="TRUE")
    ]

    task = vm.ReconfigVM_Task(spec)
    if not wait_for_task(task):
        continue

    # 3. Inject Ignition and network config
    print(f"  Injecting Ignition config...")
    with open(config["ign"], "rb") as f:
        ign_data = base64.b64encode(f.read()).decode("utf-8")

    spec = vim.vm.ConfigSpec()
    spec.extraConfig = [
        vim.option.OptionValue(
            key="guestinfo.ignition.config.data",
            value=ign_data
        ),
        vim.option.OptionValue(
            key="guestinfo.ignition.config.data.encoding",
            value="base64"
        ),
        vim.option.OptionValue(
            key="guestinfo.afterburn.initrd.network-kargs",
            value=f"ip={config['ip']}::{GATEWAY}:{NETMASK}:{config['hostname']}:ens192:off nameserver={DNS}"
        )
    ]

    task = vm.ReconfigVM_Task(spec)
    if not wait_for_task(task):
        continue

    print(f"  ✓ {vm_name} created successfully")

Disconnect(si)
print("\n=== All VMs created! ===")
print("Power on VMs in order: Bootstrap -> Masters -> Workers")
EOF

In [ ]:
python3 create_ocp_vms.py

In [ ]:
ssh -i ~/.ssh/ocp4 core@172.20.0.230

In [ ]:
./openshift-install wait-for bootstrap-complete --dir=. --log-level=info